# Fine-tune PEGASUS on SAMSum (Google Colab / Kaggle GPU)

This notebook is only a *launcher*. All logic lives in the repo's `src/summarizer` package, so the training you run
here is exactly the code that is tested in CI.

**Before you start**
1. Push the project to GitHub, then set `REPO_URL` below.
2. Colab: *Runtime → Change runtime type → T4 GPU* (free tier).
   Kaggle: *Settings → Accelerator → GPU T4 x2* and *Internet → On*.
3. Google Drive (Colab) or `/kaggle/working` (Kaggle) stores checkpoints, so a disconnect doesn't lose progress.

**What to expect.** The training loop prints its own progress bar with elapsed time and ETA. Note the real duration
for your write-up; it depends on the GPU you get. On a T4 the run uses fp32 (no native bf16), Adafactor, and gradient
checkpointing so that the 570M-parameter model fits in 16 GB.

> ⚠️ This notebook was written without access to a GPU and has **not** been executed. If a cell fails, the error
> message from the `summarizer` package should say which stage and why.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 2. Storage for checkpoints and the final model

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle")

if ON_KAGGLE:
    PERSIST = "/kaggle/working/summarizer"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/ai-content-summarizer"

os.makedirs(PERSIST, exist_ok=True)
print("Persistent storage:", PERSIST)

## 3. Get the code and install it

In [ ]:
REPO_URL = "https://github.com/<your-username>/ai-content-summarizer.git"   # <-- change me
WORKDIR = "/kaggle/working/ai-content-summarizer" if ON_KAGGLE else "/content/ai-content-summarizer"

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log -1 --oneline

In [ ]:
# Colab/Kaggle already ship a CUDA build of PyTorch, so we do NOT reinstall torch.
!pip install -q -r requirements-train.txt
!pip install -q -e . --no-deps
import torch, transformers
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__)

In [ ]:
# Point artifacts/ (models, checkpoints, reports) at persistent storage.
import pathlib, shutil
for sub in ["artifacts/models", "artifacts/checkpoints"]:
    target = pathlib.Path(PERSIST) / sub
    target.mkdir(parents=True, exist_ok=True)
    link = pathlib.Path(sub)
    if link.is_symlink():
        continue
    if link.exists():
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)
!ls -la artifacts

## 4. Data stages (ingestion → validation → transformation)
Takes about a minute. The validation report must say `passed`.

In [ ]:
!python -m summarizer.pipeline.training_pipeline --stages ingestion validation transformation
import json
print(json.load(open("artifacts/reports/validation_report.json"))["status"])

## 5. Zero-shot baseline: PEGASUS-xsum *before* fine-tuning

This scores the off-the-shelf checkpoint on the same 200 test dialogues the fine-tuned model will see. The difference
between this run and section 8 is the measurable value of fine-tuning, which makes a strong resume line. `--no-promote`
keeps this Hub model out of the registry.

In [ ]:
!python -m summarizer.pipeline.evaluation_pipeline --model google/pegasus-xsum --no-promote

## 6. Train (+ evaluate)

Pick a `RUN_ID` and **write it down**. If the session disconnects, re-run sections 1–4 (data is rebuilt; checkpoints survive on Drive), then the *resume* cell with
the same id.

Hyperparameters come from `config/params.yaml` (2 epochs, effective batch 16, lr 5e-5, Adafactor, early stopping on
`eval_loss`). To experiment, edit that file and commit it, so the run stays reproducible. The `evaluation` stage
then scores the new model on the test rows and, because it is the first real model, promotes it to production.

In [ ]:
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
print("RUN_ID =", RUN_ID)
!python -m summarizer.pipeline.training_pipeline --stages training evaluation --run-id {RUN_ID}

In [ ]:
# ONLY if the run above was interrupted: set RUN_ID to the interrupted run and resume.
# RUN_ID = "20261002-153000"
# !python -m summarizer.pipeline.training_pipeline --stages training evaluation --run-id {RUN_ID} --resume

## 7. Inspect the training run

In [ ]:
import json, pandas as pd
meta = json.load(open(f"artifacts/models/{RUN_ID}/metadata.json"))
print({k: meta[k] for k in ["train_rows", "eval_rows", "epochs_completed", "train_loss", "eval_loss",
                            "training_seconds", "bf16", "best_checkpoint"]})
print(meta["environment"])

history = pd.DataFrame(json.load(open(f"artifacts/models/{RUN_ID}/log_history.json")))
ax = history.dropna(subset=["loss"]).plot(x="step", y="loss", figsize=(8, 3), title="Training loss")
history.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o")

In [ ]:
# Quick qualitative check on a few test dialogues (full ROUGE evaluation comes in the evaluation stage).
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from summarizer.components.text_cleaning import TextCleaner

model_dir = f"artifacts/models/{RUN_ID}"
tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to("cuda").eval()
test = pd.read_csv("data/validated/test.csv").head(3)
cleaner = TextCleaner()
for _, row in test.iterrows():
    inputs = tok(cleaner.clean(row.dialogue), max_length=512, truncation=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, num_beams=4, max_length=64, no_repeat_ngram_size=3, length_penalty=0.8)
    print(row.dialogue[:300], "\n→ MODEL:", tok.decode(out[0], skip_special_tokens=True),
          "\n→ HUMAN:", row.summary, "\n")

## 8. Results: baseline vs fine-tuned, decoding experiments, CPU latency

In [ ]:
import glob
rows = []
for path in sorted(glob.glob("artifacts/reports/evaluations/*.json")):
    r = json.load(open(path))
    m = r["metrics"]
    rows.append({"model": r["model"], "profile": r["profile"], "device": r["latency"].get("device"),
                 "rows": r["eval_set"]["rows"], "ROUGE-1": m["rouge1"], "ROUGE-2": m["rouge2"],
                 "ROUGE-L": m["rougeL"], "ROUGE-Lsum": m["rougeLsum"],
                 "ROUGE-L 95% CI": m["confidence_intervals_95"]["rougeL"], "BLEU": m["bleu"],
                 "avg words": m["avg_generated_words"], "p50 latency (s)": r["latency"].get("p50_seconds")})
pd.DataFrame(rows)

In [ ]:
# Read real outputs before trusting ROUGE: the 2 worst and 2 best rows of the fine-tuned model's evaluation
# (evaluation_samples.csv was written by the evaluation stage in section 6).
samples = pd.read_csv("artifacts/reports/evaluation_samples.csv")
pd.set_option("display.max_colwidth", 200)
samples.sort_values("rougeL")[["dialogue", "generated_summary", "reference_summary", "rougeL"]].iloc[[0, 1, -2, -1]]

**Decoding experiments.** Same fine-tuned model, same 100 test rows; each row of the table changes one generation
parameter relative to the `optimized` profile. The log prints each configuration as it finishes.

In [ ]:
!python -m summarizer.pipeline.evaluation_pipeline --model {RUN_ID} --experiments
pd.read_csv("artifacts/reports/generation_experiments.csv")

**CPU latency.** The API will run on a CPU-only EC2 instance, so GPU latency would be misleading. This runs 20 test
dialogues on the CPU, one request at a time.

In [ ]:
!python -m summarizer.pipeline.evaluation_pipeline --model {RUN_ID} --device cpu -n 20 --no-promote

## 9. Bring the model and the results home

Pick **one**:

* **Hugging Face Hub (recommended).** It gives you a public model page to link from your resume, and the API/Docker
  image can download it by name (`MODEL_SOURCE=path`, `MODEL_NAME_OR_PATH=<user>/pegasus-samsum`).
* **Zip download.** The model is about 2.3 GB, which is too big for git. Keep it on Drive or upload it to S3 (Phase 6).

**Reports.** Download `reports-<RUN_ID>.zip` (created below), unzip it over your local `artifacts/reports/`, and commit
it. It holds the evaluation metrics, per-model history, experiment table, samples and training log. Every number you
put in the README or on your resume must come from these files.

In [ ]:
# Option A - push to the Hugging Face Hub (needs a WRITE token from https://huggingface.co/settings/tokens).
# from huggingface_hub import login; login()
# model.push_to_hub("<your-hf-username>/pegasus-samsum"); tok.push_to_hub("<your-hf-username>/pegasus-samsum")

In [ ]:
# Option B - zip to persistent storage, then download from Drive / Kaggle output.
!cd artifacts/models && zip -qr {PERSIST}/model-{RUN_ID}.zip {RUN_ID}
!zip -qr {PERSIST}/reports-{RUN_ID}.zip artifacts/reports artifacts/models/registry.json
!ls -lh {PERSIST}